<a href="https://colab.research.google.com/github/alvarezrodri/ProjetoPyspark/blob/main/ProjetoEbac.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark

In [2]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import DecimalType
from pyspark.sql.functions import countDistinct, col
from pyspark.ml.feature import StringIndexer, MinMaxScaler, PCA, VectorAssembler # ml - Machine learning
from pyspark.ml.clustering import KMeans
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

In [3]:
spark = SparkSession.builder.getOrCreate()

In [4]:
df_despesas_2026 = spark.read.csv(
    '/content/drive/MyDrive/ProjetoEbac/202603_Despesas.csv',
    header=True,
    inferSchema=True,
    sep=";",
    encoding="iso-8859-1"
)

In [5]:
df_despesas_2026.show(30)

+-----------------------+---------------------+--------------------+------------------------+----------------------+----------------------+--------------------+-------------+--------------------+---------------------------+-------------------------+-------------+-------------------+---------------+--------------------+----------------------------+--------------------------+-----------+--------------------+-------------------------+--------------------+-----------------------+---------------------+----+---------+----------------+--------------------+------------------+--------------------+-----------------+----------------------------------+-------------------+-----------------+--------------------------+------------------------+-----------------------+---------------------+--------------------------+------------------------+----------------------------+---------------------+--------------------+--------------------+---------------+-----------------------------------+-------------------

In [6]:
# Colunas que eu vou utilizar
colunas_utilizadas = [
    "Ano e mês do lançamento",
    "Código Órgão Superior","Nome Órgão Superior",
    "Código Órgão Subordinado", "Nome Órgão Subordinado",
    "Código Unidade Gestora", "Nome Unidade Gestora",
    "Código Elemento de Despesa", "Nome Elemento de Despesa",
    "Modalidade da Despesa",
    "Valor Empenhado (R$)", "Valor Liquidado (R$)", "Valor Pago (R$)"

]

In [7]:
df_despesas_reduzido = df_despesas_2026.select(colunas_utilizadas)

In [8]:
df_despesas_reduzido.show(10)

+-----------------------+---------------------+--------------------+------------------------+----------------------+----------------------+--------------------+--------------------------+------------------------+---------------------+--------------------+--------------------+---------------+
|Ano e mês do lançamento|Código Órgão Superior| Nome Órgão Superior|Código Órgão Subordinado|Nome Órgão Subordinado|Código Unidade Gestora|Nome Unidade Gestora|Código Elemento de Despesa|Nome Elemento de Despesa|Modalidade da Despesa|Valor Empenhado (R$)|Valor Liquidado (R$)|Valor Pago (R$)|
+-----------------------+---------------------+--------------------+------------------------+----------------------+----------------------+--------------------+--------------------------+------------------------+---------------------+--------------------+--------------------+---------------+
|                2026/03|                26000|Ministério da Edu...|                   26243|  Universidade Fede...|     

In [9]:
df_despesas_reduzido.printSchema()

root
 |-- Ano e mês do lançamento: string (nullable = true)
 |-- Código Órgão Superior: integer (nullable = true)
 |-- Nome Órgão Superior: string (nullable = true)
 |-- Código Órgão Subordinado: integer (nullable = true)
 |-- Nome Órgão Subordinado: string (nullable = true)
 |-- Código Unidade Gestora: integer (nullable = true)
 |-- Nome Unidade Gestora: string (nullable = true)
 |-- Código Elemento de Despesa: integer (nullable = true)
 |-- Nome Elemento de Despesa: string (nullable = true)
 |-- Modalidade da Despesa: string (nullable = true)
 |-- Valor Empenhado (R$): string (nullable = true)
 |-- Valor Liquidado (R$): string (nullable = true)
 |-- Valor Pago (R$): string (nullable = true)



In [10]:
df_despesas_reduzido = df_despesas_reduzido\
    .withColumnRenamed("Ano e mês do lançamento", "ano_mes_lancamento") \
    .withColumnRenamed("Código Órgão Superior", "codigo_orgao_superior") \
    .withColumnRenamed("Nome Órgão Superior", "nome_orgao_superior") \
    .withColumnRenamed("Código Órgão Subordinado", "codigo_orgao_subordinado") \
    .withColumnRenamed("Nome Órgão Subordinado", "nome_orgao_subordinado") \
    .withColumnRenamed("Código Unidade Gestora", "codigo_unidade_gestora") \
    .withColumnRenamed("Nome Unidade Gestora", "nome_unidade_gestora") \
    .withColumnRenamed("Código Elemento de Despesa", "codigo_elemento_despesa") \
    .withColumnRenamed("Nome Elemento de Despesa", "nome_elemento_despesa") \
    .withColumnRenamed("Modalidade da Despesa", "modalidade_despesa") \
    .withColumnRenamed("Valor Empenhado (R$)", "valor_empenhado_(R$)") \
    .withColumnRenamed("Valor Liquidado (R$)", "valor_liquidado_(R$)") \
    .withColumnRenamed("Valor Pago (R$)", "valor_pago_(R$)")


In [11]:
df_despesas_reduzido.show()

df_despesas_reduzido.printSchema()

+------------------+---------------------+--------------------+------------------------+----------------------+----------------------+--------------------+-----------------------+---------------------+------------------+--------------------+--------------------+---------------+
|ano_mes_lancamento|codigo_orgao_superior| nome_orgao_superior|codigo_orgao_subordinado|nome_orgao_subordinado|codigo_unidade_gestora|nome_unidade_gestora|codigo_elemento_despesa|nome_elemento_despesa|modalidade_despesa|valor_empenhado_(R$)|valor_liquidado_(R$)|valor_pago_(R$)|
+------------------+---------------------+--------------------+------------------------+----------------------+----------------------+--------------------+-----------------------+---------------------+------------------+--------------------+--------------------+---------------+
|           2026/03|                26000|Ministério da Edu...|                   26243|  Universidade Fede...|                153103|UNIVERSIDADE FEDE...|        

In [12]:
tipo_decimal = DecimalType(15, 2)

In [13]:
# Remove os pontos da Strings ou separador de milhares "."
df_despesas_reduzido = df_despesas_reduzido \
    .withColumn("valor_empenhado_(R$)", regexp_replace(col("valor_empenhado_(R$)"), "\\.", "")) \
    .withColumn("valor_liquidado_(R$)", regexp_replace(col("valor_liquidado_(R$)"), "\\.", "")) \
    .withColumn("valor_pago_(R$)", regexp_replace(col("valor_pago_(R$)"), "\\.", ""))

# Tranformar string e tipo dedimal do "tipo_decimal" e troca "," para "."
df_despesas_reduzido = df_despesas_reduzido \
    .withColumn("valor_empenhado_(R$)", regexp_replace(col("valor_empenhado_(R$)"), ",", ".").cast("float")) \
    .withColumn("valor_liquidado_(R$)", regexp_replace(col("valor_liquidado_(R$)"), ",", ".").cast("float")) \
    .withColumn("valor_pago_(R$)", regexp_replace(col("valor_pago_(R$)"), ",", ".").cast("float"))

##Tratando valores nulos

In [14]:
# Tratar valores nulos dessas colunas
colunas_obrigatorias = [
    "ano_mes_lancamento",
    "nome_orgao_superior",
    "nome_unidade_gestora",
    "nome_elemento_despesa",
    "valor_pago_(R$)"
]

df_despesas_reduzido = df_despesas_reduzido.dropna(subset=colunas_obrigatorias)

In [15]:
# Tratando valores nulos em colunas numericas
df_despesas_reduzido = df_despesas_reduzido.fillna({
    "valor_empenhado_(R$)": 0.0,
    "valor_liquidado_(R$)": 0.0,
    "valor_pago_(R$)": 0.0
})

In [16]:
df_despesas_reduzido = df_despesas_reduzido.fillna({
    "modalidade_despesa":"Não Informado",
    "nome_orgao_subordinado":"Não Informado"
})

In [17]:
df_despesas_reduzido.show()

+------------------+---------------------+--------------------+------------------------+----------------------+----------------------+--------------------+-----------------------+---------------------+------------------+--------------------+--------------------+---------------+
|ano_mes_lancamento|codigo_orgao_superior| nome_orgao_superior|codigo_orgao_subordinado|nome_orgao_subordinado|codigo_unidade_gestora|nome_unidade_gestora|codigo_elemento_despesa|nome_elemento_despesa|modalidade_despesa|valor_empenhado_(R$)|valor_liquidado_(R$)|valor_pago_(R$)|
+------------------+---------------------+--------------------+------------------------+----------------------+----------------------+--------------------+-----------------------+---------------------+------------------+--------------------+--------------------+---------------+
|           2026/03|                26000|Ministério da Edu...|                   26243|  Universidade Fede...|                153103|UNIVERSIDADE FEDE...|        

In [18]:
df_despesas_reduzido = df_despesas_reduzido.withColumn(
    "data_formatada", to_date(col("ano_mes_lancamento"), "yyyy/MM")
)

In [19]:
df_despesas_reduzido = df_despesas_reduzido.withColumn("ano", date_format(col("data_formatada"), 'yyyy'))
df_despesas_reduzido = df_despesas_reduzido.withColumn("mes", date_format(col("data_formatada"), 'MM'))
df_despesas_reduzido.select("data_formatada").distinct().show(10, truncate=False)


+--------------+
|data_formatada|
+--------------+
|2026-03-01    |
+--------------+



In [20]:
df_despesas_reduzido.show()
df_despesas_reduzido.printSchema()

+------------------+---------------------+--------------------+------------------------+----------------------+----------------------+--------------------+-----------------------+---------------------+------------------+--------------------+--------------------+---------------+--------------+----+---+
|ano_mes_lancamento|codigo_orgao_superior| nome_orgao_superior|codigo_orgao_subordinado|nome_orgao_subordinado|codigo_unidade_gestora|nome_unidade_gestora|codigo_elemento_despesa|nome_elemento_despesa|modalidade_despesa|valor_empenhado_(R$)|valor_liquidado_(R$)|valor_pago_(R$)|data_formatada| ano|mes|
+------------------+---------------------+--------------------+------------------------+----------------------+----------------------+--------------------+-----------------------+---------------------+------------------+--------------------+--------------------+---------------+--------------+----+---+
|           2026/03|                26000|Ministério da Edu...|                   26243|  U

# Análise explorátoria (ADA)

In [21]:
df_despesas_reduzido.agg(countDistinct(col("nome_orgao_superior")).alias("qtd_orgao_superior_distintos")).show()

+----------------------------+
|qtd_orgao_superior_distintos|
+----------------------------+
|                          35|
+----------------------------+



In [22]:
# Ranqueando o orgão que gastou mais
ranking_orgao_superior = df_despesas_reduzido.groupBy("nome_orgao_superior").agg(
    round((sum("valor_pago_(R$)") / 1_000_000_000), 2).alias("total_pago_bilhoes")
).orderBy("total_pago_bilhoes", ascending=False)

ranking_orgao_superior.show(truncate=False)

+--------------------------------------------+------------------+
|nome_orgao_superior                         |total_pago_bilhoes|
+--------------------------------------------+------------------+
|Ministério da Fazenda                       |882.89            |
|Ministério da Previdência Social            |160.95            |
|Ministério da Educação                      |30.59             |
|Ministério da Saúde                         |26.86             |
|Ministério da Defesa                        |15.18             |
|Ministério do Desenvolvimento e Assistência |14.92             |
|Ministério do Trabalho e Emprego            |11.62             |
|Ministério das Cidades                      |5.28              |
|Ministério da Gestão e da Inovação em Ser   |4.53              |
|Ministério da Integração e do Desenvolvime  |3.59              |
|Ministério da Justiça e Segurança Pública   |2.44              |
|Ministério de Minas e Energia               |1.88              |
|Ministéri

In [23]:
# Por unidade gestora
unidade_gestora_ranking = df_despesas_reduzido.groupBy("nome_unidade_gestora") \
    .agg(round((sum("valor_pago_(R$)") / 1_000_000_000), 2).alias("total_pago_bilhoes")) \
    .orderBy(col("total_pago_bilhoes").desc())

unidade_gestora_ranking.show(truncate= False)

+---------------------------------------------+------------------+
|nome_unidade_gestora                         |total_pago_bilhoes|
+---------------------------------------------+------------------+
|COORD.GERAL DE CONTROLE DA DIVIDA PUBLICA    |856.01            |
|COORDENACAO DE ORCAMENTO E FINANCAS DO FRGPS |139.17            |
|DIRETORIA EXECUTIVA DO FUNDO NAC. DE SAUDE   |20.44             |
|COORD.-GERAL DE TRANSFE. INTERGOVERNAMENTAIS |20.13             |
|COORD.GERAL DE ORCAMENTO, FINANCAS E CONTAB. |19.48             |
|FUNDO NACIONAL DE DESENVOLVIMENTO DA EDUCACAO|13.63             |
|SECRETARIA NACIONAL DE RENDA DA CIDADANIA    |12.81             |
|COORD-GERAL DE RECURSOS DO FAT - CGFAT       |11.35             |
|CENTRO DE PAGAMENTO DO EXERCITO              |6.69              |
|CEF - FUNDO SOCIAL/MCID                      |4.28              |
|PAGADORIA DE PESSOAL DA MARINHA - PAPEM-PAIS |4.09              |
|COORDENACAO-GERAL DE GESTAO DE PESSOAS       |3.8            

In [24]:
from os import truncate
tipos_despesa = df_despesas_reduzido.groupBy("nome_elemento_despesa") \
    .agg(round((sum("valor_pago_(R$)") / 1_000_000_000), 2).alias("total_pago_bilhoes")) \
    .orderBy(col("total_pago_bilhoes").desc())

tipos_despesa.show(truncate=False)

+---------------------------------------------+------------------+
|nome_elemento_despesa                        |total_pago_bilhoes|
+---------------------------------------------+------------------+
|Principal Corrigido da Dívida Mobiliária Re  |690.44            |
|Juros, Deságios e Descontos da Dívida Mobil  |163.8             |
|Aposentadorias do RGPS - Área Urbana         |76.23             |
|Contribuições                                |25.36             |
|Pensões do RGPS - Área Urbana                |23.61             |
|Distribuição Constitucional ou Legal de Rec  |23.46             |
|Aposentadorias do RGPS - Área Rural          |21.53             |
|Benefício Mensal ao Deficiente e ao Idoso    |19.25             |
|Vencimentos e Vantagens Fixas - Pessoal Civil|16.39             |
|Aposentadorias, Reserva Remunerada e Reformas|15.55             |
|Outros Auxílios Financeiros a Pessoas Físic  |14.08             |
|Despesas de Exercícios Anteriores            |9.73           

## Preprocessamento de dados




In [25]:
df_despesas_reduzido.show()

+------------------+---------------------+--------------------+------------------------+----------------------+----------------------+--------------------+-----------------------+---------------------+------------------+--------------------+--------------------+---------------+--------------+----+---+
|ano_mes_lancamento|codigo_orgao_superior| nome_orgao_superior|codigo_orgao_subordinado|nome_orgao_subordinado|codigo_unidade_gestora|nome_unidade_gestora|codigo_elemento_despesa|nome_elemento_despesa|modalidade_despesa|valor_empenhado_(R$)|valor_liquidado_(R$)|valor_pago_(R$)|data_formatada| ano|mes|
+------------------+---------------------+--------------------+------------------------+----------------------+----------------------+--------------------+-----------------------+---------------------+------------------+--------------------+--------------------+---------------+--------------+----+---+
|           2026/03|                26000|Ministério da Edu...|                   26243|  U

## transformar nome_orgao_superior, nome_elemento_despesa e modalidade_despesa  em númerico para meu modelo

In [26]:

df_despesas_reduzido = df_despesas_reduzido.drop("nome_orgao_superior_index")

# Indexar orgao_superior
indexador = StringIndexer(inputCol="nome_orgao_superior", outputCol="nome_orgao_superior_index")
modelo_indexar = indexador.fit(df_despesas_reduzido)
df_indexado = modelo_indexar.transform(df_despesas_reduzido)  # salva em df_indexado

# Indexar nome_elemento_despesa
indexador_elemento = StringIndexer(inputCol="nome_elemento_despesa", outputCol="nome_elemento_despesa_index")
modelo_elemento = indexador_elemento.fit(df_indexado)
df_indexado = modelo_elemento.transform(df_indexado)  # acumula no mesmo df

# Indexar modalidade_despesa
indexador_modalidade = StringIndexer(inputCol="modalidade_despesa", outputCol="modalidade_despesa_index")
modelo_modalidade = indexador_modalidade.fit(df_indexado)
df_indexado = modelo_modalidade.transform(df_indexado)  # acumula no mesmo df

df_indexado.show()

+------------------+---------------------+--------------------+------------------------+----------------------+----------------------+--------------------+-----------------------+---------------------+------------------+--------------------+--------------------+---------------+--------------+----+---+-------------------------+---------------------------+------------------------+
|ano_mes_lancamento|codigo_orgao_superior| nome_orgao_superior|codigo_orgao_subordinado|nome_orgao_subordinado|codigo_unidade_gestora|nome_unidade_gestora|codigo_elemento_despesa|nome_elemento_despesa|modalidade_despesa|valor_empenhado_(R$)|valor_liquidado_(R$)|valor_pago_(R$)|data_formatada| ano|mes|nome_orgao_superior_index|nome_elemento_despesa_index|modalidade_despesa_index|
+------------------+---------------------+--------------------+------------------------+----------------------+----------------------+--------------------+-----------------------+---------------------+------------------+----------------

Neste trecho do código, foram preparados alguns dados para que pudessem ser utilizados em modelos de machine learning no PySpark. Primeiro, foi removida uma coluna antiga de índice do órgão para evitar duplicidade. Em seguida, algumas colunas que possuem informações em formato de texto, como o nome do órgão superior, o elemento da despesa e a modalidade da despesa, foram convertidas para valores numéricos. Essa transformação é necessária porque os algoritmos de aprendizado de máquina não conseguem trabalhar diretamente com textos. Assim, cada categoria dessas colunas passou a ser representada por um número correspondente. O resultado final é um conjunto de dados mais adequado para a aplicação de modelos de análise e aprendizado de máquina.

Características

In [27]:
features = [
    "nome_orgao_superior_index",
    "codigo_elemento_despesa",
    "nome_elemento_despesa_index",
    "modalidade_despesa_index",
    "valor_empenhado_(R$)",
    "valor_liquidado_(R$)"
]

In [28]:
assembler = VectorAssembler(inputCols=features, outputCol="features_raw")
df_assembled = assembler.transform(df_indexado)

# Normalizar os valores numéricos com MinMaxScaler
scaler = MinMaxScaler(inputCol="features_raw", outputCol="features")
scaler_model = scaler.fit(df_assembled)
df_scaled = scaler_model.transform(df_assembled)

df_scaled.show()

+------------------+---------------------+--------------------+------------------------+----------------------+----------------------+--------------------+-----------------------+---------------------+------------------+--------------------+--------------------+---------------+--------------+----+---+-------------------------+---------------------------+------------------------+--------------------+--------------------+
|ano_mes_lancamento|codigo_orgao_superior| nome_orgao_superior|codigo_orgao_subordinado|nome_orgao_subordinado|codigo_unidade_gestora|nome_unidade_gestora|codigo_elemento_despesa|nome_elemento_despesa|modalidade_despesa|valor_empenhado_(R$)|valor_liquidado_(R$)|valor_pago_(R$)|data_formatada| ano|mes|nome_orgao_superior_index|nome_elemento_despesa_index|modalidade_despesa_index|        features_raw|            features|
+------------------+---------------------+--------------------+------------------------+----------------------+----------------------+------------------

#Utilizando Modelos de machine learning

###clusterização(KMeans) Par agrupar gastos similares

In [29]:
assembler = VectorAssembler(inputCols=features, outputCol="features_raw")
df_assembled = assembler.transform(df_indexado)

# Normalizar os valores numéricos com MinMaxScaler
scaler = MinMaxScaler(inputCol="features_raw", outputCol="features")
scaler_model = scaler.fit(df_assembled)
df_scaled = scaler_model.transform(df_assembled)

df_scaled.show()

+------------------+---------------------+--------------------+------------------------+----------------------+----------------------+--------------------+-----------------------+---------------------+------------------+--------------------+--------------------+---------------+--------------+----+---+-------------------------+---------------------------+------------------------+--------------------+--------------------+
|ano_mes_lancamento|codigo_orgao_superior| nome_orgao_superior|codigo_orgao_subordinado|nome_orgao_subordinado|codigo_unidade_gestora|nome_unidade_gestora|codigo_elemento_despesa|nome_elemento_despesa|modalidade_despesa|valor_empenhado_(R$)|valor_liquidado_(R$)|valor_pago_(R$)|data_formatada| ano|mes|nome_orgao_superior_index|nome_elemento_despesa_index|modalidade_despesa_index|        features_raw|            features|
+------------------+---------------------+--------------------+------------------------+----------------------+----------------------+------------------

In [30]:
assembler = VectorAssembler(inputCols=features, outputCol="features_raw")
df_assembled = assembler.transform(df_indexado)

# Normalizar os valores numéricos com MinMaxScaler
scaler = MinMaxScaler(inputCol="features_raw", outputCol="features")
scaler_model = scaler.fit(df_assembled)
df_scaled = scaler_model.transform(df_assembled)

df_scaled.show()

+------------------+---------------------+--------------------+------------------------+----------------------+----------------------+--------------------+-----------------------+---------------------+------------------+--------------------+--------------------+---------------+--------------+----+---+-------------------------+---------------------------+------------------------+--------------------+--------------------+
|ano_mes_lancamento|codigo_orgao_superior| nome_orgao_superior|codigo_orgao_subordinado|nome_orgao_subordinado|codigo_unidade_gestora|nome_unidade_gestora|codigo_elemento_despesa|nome_elemento_despesa|modalidade_despesa|valor_empenhado_(R$)|valor_liquidado_(R$)|valor_pago_(R$)|data_formatada| ano|mes|nome_orgao_superior_index|nome_elemento_despesa_index|modalidade_despesa_index|        features_raw|            features|
+------------------+---------------------+--------------------+------------------------+----------------------+----------------------+------------------

In [31]:
kmeans = KMeans(featuresCol="features", k=5, seed=42)
modelo_kmeans = kmeans.fit(df_scaled)
df_clusters = modelo_kmeans.transform(df_scaled)
df_clusters.groupBy("prediction", "nome_orgao_superior").count().orderBy("prediction").show()

+----------+--------------------+-----+
|prediction| nome_orgao_superior|count|
+----------+--------------------+-----+
|         0|Ministério do Des...| 1467|
|         0|Ministério de Min...|  975|
|         0|Ministério da Edu...|11341|
|         0|Ministério da Jus...| 1734|
|         0|Ministério da Defesa|13534|
|         0| Ministério da Saúde| 2221|
|         0|Ministério das Re...| 1447|
|         0|Ministério da Agr...| 2010|
|         0|Ministério da Int...|  444|
|         0|Ministério da Ges...| 1383|
|         1|Ministério da Edu...| 5396|
|         1|Ministério da Defesa| 1877|
|         1|Ministério da Int...|  181|
|         1|Ministério do Des...|  594|
|         1| Ministério da Saúde|  333|
|         1|Ministério da Jus...|  298|
|         1|Ministério dos Tr...|   10|
|         1|Ministério da Faz...|   32|
|         1|Ministério da Ges...|  151|
|         1|Ministério da Ciê...|   91|
+----------+--------------------+-----+
only showing top 20 rows


Aplicação do algoritmo K-Means para identificar padrões de despesas.
O modelo agrupa registros semelhantes em 5 clusters com base nas variáveis da coluna "features".
Após o treinamento do modelo, cada registro recebe um cluster (coluna "prediction").
Em seguida, é feita uma análise agrupando os clusters pelo nome do órgão superior,
mostrando quantas despesas de cada órgão pertencem a cada grupo identificado pelo modelo.

Média de valor pago por cada cluster

In [32]:
df_clusters.groupBy("prediction").avg("valor_pago_(R$)").show()

+----------+--------------------+
|prediction|avg(valor_pago_(R$))|
+----------+--------------------+
|         1|2.1792525091165233E7|
|         3|  5106630.3624363495|
|         4|  2818400.4747739984|
|         2| 6.575760084502218E8|
|         0|   831564.9627616643|
+----------+--------------------+



##Regressão para prever valor_pago_(R$)

In [33]:
train, test = df_scaled.randomSplit([0.8, 0.2], seed=42)

rf = RandomForestRegressor(featuresCol="features", labelCol="valor_pago_(R$)", numTrees=50)
modelo_rf = rf.fit(train)
predicoes = modelo_rf.transform(test)

evaluator = RegressionEvaluator(labelCol="valor_pago_(R$)", predictionCol="prediction", metricName="rmse")
print(f"RMSE: {evaluator.evaluate(predicoes):.2f}")

RMSE: 6226011644.95


O modelo apresentou um RMSE de aproximadamente 6,2 bilhões de reais, indicando que o erro médio das previsões é elevado. Esse comportamento pode estar relacionado à alta variabilidade dos valores de despesas públicas e à presença de valores extremos na base de dados.

## PCA Reduzir dimensionalidade para visualizar os clusters ou entender as features mais importantes:

In [34]:
pca = PCA(k=2, inputCol="features", outputCol="pca_features")
modelo_pca = pca.fit(df_scaled)
df_pca = modelo_pca.transform(df_scaled)
print(f"Variância explicada: {modelo_pca.explainedVariance}")

Variância explicada: [0.47129560079436883,0.3395966112407453]


Foi aplicada a técnica de redução de dimensionalidade PCA com dois componentes principais. O primeiro componente explicou aproximadamente 47% da variância dos dados, enquanto o segundo explicou cerca de 34%. No total, os dois componentes capturaram aproximadamente 81% da variabilidade do conjunto de dados, indicando uma boa representação das informações originais.

#Salvando

In [35]:
df_clusters.write.parquet("/content/drive/MyDrive/ProjetoEbac/output/despesas_202603.parquet")

In [36]:
df_despesas_reduzido.write.parquet("/content/drive/MyDrive/ProjetoEbac/output/despesas_202603_tratamento.parquet")

In [37]:
spark.stop()